# HW6 Q1: VGG16 hardware mapping with tiling
Map the 3rd layer of the 4-bit VGG16 (`model.features[3]`, a 64 → 64 channel 3×3 QuantConv2d on a 32×32 feature map) onto a **16×16 weight-stationary systolic array**: each array row is an input channel and each column an output channel, so the layer is split into 4 ic tiles × 4 oc tiles = 16 tiles. Then check that the tiled computation reproduces the layer output.

**For the submission:** after training, do *Kernel → Restart & Clear Output*, then run every cell **except** the training cell.

## Setup

In [ ]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "VGG16_quant"
model = VGG16_quant()   # 4-bit QuantConv2d in every conv layer (models/vgg_quant.py)

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

## 1. 4-bit quantization-aware training (run once, then Restart & Clear Output; skip this cell for the submission)

In [ ]:
# 4-bit QAT training of VGG16_quant. Run this cell ONCE to train and save the best checkpoint to
# result/VGG16_quant/. Then do Kernel -> Restart & Clear Output, and for the submission run every cell EXCEPT this one.

# Start from the trained floating-point VGG16 of HW3 (93.17%): VGG16_quant has the same layers in the same
# order, so its conv, BatchNorm and FC weights load directly. QuantConv2d has extra parameters (act_alpha,
# wgt_alpha, weight_q) that the HW3 model does not have, so load with strict=False and keep their defaults.
init_from_hw3 = True
hw3_path = 'result/vgg16/model_best.pth.tar'

if init_from_hw3:
    lr, epochs = 0.02, 50        # fine-tuning: a lower lr keeps the trained weights
    state_hw3 = torch.load(hw3_path, map_location=device)['state_dict']
    missing, unexpected = model.load_state_dict(state_hw3, strict=False)
    print('loaded from HW3:', len(state_hw3) - len(unexpected), 'tensors.  Not in HW3 (kept at default):',
          sorted(set(k.split('.')[-1] for k in missing)), ' unused:', unexpected)
else:
    lr, epochs = 0.05, 100       # from scratch (a 4-bit VGG16 diverges easily at lr 0.1)

# The first layer's input is the normalized image (about -2 to 2.1). With the default act_alpha = 8.0 the
# 4-bit step is 8/15 = 0.53, so the image would use only about 8 levels, and act_alpha would never train
# (its gradient comes only from clipped inputs, and nothing is above 8). Start it at 3.0: step 0.2.
with torch.no_grad():
    model.features[0].act_alpha.fill_(3.0)

weight_decay = 5e-4
best_prec = 0

model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)

# Same optimizer setup as HW5: the clipping values (alphas) get no weight decay, and weight_q (a copy of the
# last quantized weights that forward() stores) is not optimized.
alpha_params = [p for n, p in model.named_parameters() if 'alpha' in n]
other_params = [p for n, p in model.named_parameters() if 'alpha' not in n and 'weight_q' not in n]
optimizer = torch.optim.SGD([{'params': other_params, 'weight_decay': weight_decay},
                             {'params': alpha_params, 'weight_decay': 0}],
                            lr=lr, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)

for epoch in range(0, epochs):
    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    scheduler.step()

## Test accuracy of the trained 4-bit model

In [ ]:
# Reload the saved best checkpoint and evaluate it on the test set.
fdir = 'result/'+str(model_name)+'/model_best.pth.tar'
checkpoint = torch.load(fdir, map_location=device)
model.load_state_dict(checkpoint['state_dict'])

criterion = nn.CrossEntropyLoss().to(device)
model.to(device)
model.eval()

prec = validate(testloader, model, criterion)
print('4-bit VGG16_quant test accuracy: {:.2f}%'.format(prec))
model.show_params()   # trained clipping values (alpha) of every QuantConv2d

## 2. Target layer: `model.features[3]`
Counting every module in `model.features` from 0: `[0]` first conv, `[1]` BatchNorm, `[2]` ReLU, so `[3]` is the second conv layer (64 → 64 channels).

In [ ]:
print(model.features[:4])
layer = model.features[3]
print('target layer:', layer, '| bit width:', layer.bit)

## 3. Integer inputs `x_int` and integer weights `w_int`
Same steps as HW5: hook the layer's input, quantize it with the layer's own activation quantizer and clipping value, and divide the quantized values by their step sizes. Only one image is used.

In [ ]:
hooked = {}
def save_input(module, input):
    hooked['x'] = input[0].detach().clone()     # clone: the ReLU before this layer is in-place
def save_output(module, input, output):
    hooked['out'] = output.detach().clone()

handles = [layer.register_forward_pre_hook(save_input),
           layer.register_forward_hook(save_output)]

images, labels = next(iter(testloader))
model.eval()
with torch.no_grad():
    model(images[:1].to(device))   # one image; forward() also stores this layer's quantized weights in layer.weight_q
for h in handles:
    h.remove()

w_bit, x_bit = 4, 4
w_alpha = layer.weight_quant.wgt_alpha.detach()
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)          # signed weights: -7..7
x_alpha = layer.act_alpha.detach()
x_delta = x_alpha / (2 ** x_bit - 1)                # unsigned activations: 0..15

w_int = layer.weight_q.detach() / w_delta           # weight_q (normalized units), not layer.weight
with torch.no_grad():
    x_int = act_quantization(x_bit)(hooked['x'], x_alpha) / x_delta
print('max distance from an integer:  w', (w_int - w_int.round()).abs().max().item(),
      '  x', (x_int - x_int.round()).abs().max().item())

# round() only removes float noise. Move to the CPU: the tiling below runs many small operations.
w_int = w_int.round().cpu()                          # [C_out, C_in, k, k]
x_int = x_int.round()[0].cpu()                       # [C_in, H, W] (one image)
print('w_int:', list(w_int.shape), ' values', int(w_int.min()), 'to', int(w_int.max()))
print('x_int:', list(x_int.shape), ' values', int(x_int.min()), 'to', int(x_int.max()))

## 4. Tiling the weights
The 16×16 array holds a 16 (ic) × 16 (oc) weight block for one kernel pixel `kij = ky·3 + kx`. With 64 input and 64 output channels there are 4 ic tiles × 4 oc tiles. In tile (ic_tile, oc_tile), the PE at row `r` and column `c` holds

&emsp;&emsp;`w_int[16·oc_tile + c, 16·ic_tile + r, ky, kx]`

`w_tile` stores these blocks as [ic_tile, oc_tile, row r (ic), column c (oc), kij].

In [ ]:
array_size = 16                                      # 16x16 systolic array
C_out, C_in, ki_dim, kj_dim = w_int.shape            # 64, 64, 3, 3
n_ic_tile = C_in // array_size                       # 4
n_oc_tile = C_out // array_size                      # 4
n_kij = ki_dim * kj_dim                              # 9

w_tile = torch.zeros(n_ic_tile, n_oc_tile, array_size, array_size, n_kij)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        for kij in range(n_kij):
            ky, kx = kij // kj_dim, kij % kj_dim
            for r in range(array_size):              # array row    = input channel
                for c in range(array_size):          # array column = output channel
                    w_tile[ic_tile, oc_tile, r, c, kij] = w_int[array_size*oc_tile + c, array_size*ic_tile + r, ky, kx]

print('w_tile [ic_tile, oc_tile, row (ic), col (oc), kij]:', list(w_tile.shape))

## 5. Input tile `a_tile`
Zero-pad `x_int` to 34×34 (in the integer domain) and flatten the pixels: `nij = y·34 + x` is the time step at which a pixel enters the array. Every tile streams the whole padded image, and the same activations are streamed again for each oc tile (only 16 output channels are resident at a time), so `a_tile` has an oc_tile dimension even though its copies are identical.

In [ ]:
padding = 1
a_pad = F.pad(x_int, (padding, padding, padding, padding))   # [64, 34, 34], zeros around the border
pad_dim = a_pad.size(1)                                      # 34
a_pad = a_pad.reshape(C_in, -1)                              # [64, 1156], column nij = y*34 + x
n_nij = a_pad.size(1)                                        # 1156 time steps

a_tile = torch.zeros(n_ic_tile, n_oc_tile, array_size, n_nij)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        a_tile[ic_tile, oc_tile] = a_pad[array_size*ic_tile : array_size*(ic_tile+1), :]   # 16 rows (ic) of this tile

print('a_tile [ic_tile, oc_tile, ic index (row #), nij (time step)]:', list(a_tile.shape))

## 6. Partial sums `psum`
At each time step `nij`, the 16 activations entering the rows are multiplied by the 16×16 weight block, and each column `c` accumulates down its rows:

&emsp;&emsp;`psum[ic_tile, oc_tile, c, nij, kij] = Σ_r  w_tile[ic_tile, oc_tile, r, c, kij] · a_tile[ic_tile, oc_tile, r, nij]`  (sum over the 16 rows r)

For all 1156 time steps at once, this is one matrix multiplication per tile and kij: Wᵀ (16 oc × 16 ic) times the 16 × 1156 input tile.

In [ ]:
psum = torch.zeros(n_ic_tile, n_oc_tile, array_size, n_nij, n_kij)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        for kij in range(n_kij):
            W = w_tile[ic_tile, oc_tile, :, :, kij]                       # [16 rows (ic), 16 cols (oc)]
            psum[ic_tile, oc_tile, :, :, kij] = W.t() @ a_tile[ic_tile, oc_tile]   # [16 oc, 1156 nij]

print('psum [ic_tile, oc_tile, oc index (col #), nij (time step), kij]:', list(psum.shape))

# The same values, one time step at a time as the array produces them (shown for tile (0, 0), kij 0):
W = w_tile[0, 0, :, :, 0]
psum_steps = torch.stack([W.t() @ a_tile[0, 0, :, nij] for nij in range(n_nij)], dim=1)   # 16x16 matrix-vector per step
print('time-step version matches the matrix multiplication:', torch.equal(psum_steps, psum[0, 0, :, :, 0]))

## 7. Accumulate into the output `out`
Output pixel (oy, ox), `o_nij = oy·32 + ox`, at kernel pixel (ky, kx) uses the padded input pixel (oy + ky, ox + kx), i.e. the psum at time step

&emsp;&emsp;`nij = (oy + ky)·34 + (ox + kx)`

Sum these over all 9 kij and all 4 ic tiles. Output channel `16·oc_tile + c` comes from column `c` of oc tile `oc_tile`. Psums at the other nij (near the padded border) never map to a valid output and are not used.

In [ ]:
o_dim = pad_dim - (ki_dim - 1)                     # 34 - 2 = 32 (stride 1)
n_o_nij = o_dim * o_dim                            # 1024 output pixels

out = torch.zeros(C_out, n_o_nij)
for o_nij in range(n_o_nij):
    oy, ox = o_nij // o_dim, o_nij % o_dim
    for kij in range(n_kij):
        ky, kx = kij // kj_dim, kij % kj_dim
        nij = (oy + ky) * pad_dim + (ox + kx)       # time step of the matching input pixel
        for oc_tile in range(n_oc_tile):
            for ic_tile in range(n_ic_tile):
                out[array_size*oc_tile : array_size*(oc_tile+1), o_nij] += psum[ic_tile, oc_tile, :, nij, kij]

print('out [oc index, o_nij]:', list(out.shape))

## 8. Check against the integer reference
A regular convolution of the same integers, `conv(x_int, w_int)`, reshaped to [64, 1024]. The mapped result must match it exactly, so the absolute difference should be 0.

In [ ]:
conv_int = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)
conv_int.weight = nn.Parameter(w_int)
with torch.no_grad():
    out_ref = conv_int(x_int.unsqueeze(0)).reshape(C_out, -1)   # [64, 1024]

print('shapes:  a_tile', list(a_tile.shape), '  psum', list(psum.shape), '  out', list(out.shape))
print('absolute difference |out - conv(x_int, w_int)|:  sum =', (out - out_ref).abs().sum().item(),
      '  max =', (out - out_ref).abs().max().item())

For reference: rescaled by the step sizes, the mapped integer result also matches the layer's own (floating-point) output captured by the hook, up to float rounding.

In [ ]:
out_recovered = out * x_delta.cpu() * w_delta.cpu()
layer_out = hooked['out'][0].cpu().reshape(C_out, -1)
print('|out x delta_x x delta_w - layer output|: mean =', (out_recovered - layer_out).abs().mean().item())